# Train DistilBERT 3-Class Issue Classifier (Maintainer's Copilot)

**End-to-end DL workflow — fetch, label-merge, split, train, model card, verify.**

Runs on Colab (T4 GPU, free tier). Produces `artifacts/classifier/best/` containing weights + tokenizer + `model_card.json`, ready to be loaded by `app.ml.classifier.load_classifier()` in the backend.

## Why this is a notebook, not Python in the repo
Training is offline, runs once on a GPU. Inference is online, runs in production. The contract between the two is `model_card.json` — the only file that crosses the boundary. The backend has zero training-side imports.

## Required contract with backend (`app/ml/classifier.py`)
The model card MUST contain:
* `classes`: `["bug", "feature", "support"]` (exact order — drift = refuse to boot)
* `model_sha256`: `sha256:<hex>` of all weight files sorted by name
* `version`: semver string
* `hyperparameters.max_length`: int (used by tokenizer at inference)

## Dataset rationale (also in `docs/DECISIONS.md`)
MONAI closed-issue label counts: `bug=337`, `feature_request=535`, `documentation=28`, `questions=250`.
A 28-example `documentation` class yields a ~5-example test slice — F1 on 5 examples is noise (1 error ≈ 20-pt swing).
Merge `documentation + questions → support` (maintainer's routing is identical for both). Final balanced 3-class problem: `bug=337`, `feature=535`, `support=278`.

## 0. Setup — install deps, mount Drive

In [1]:
# Pin to versions used in backend/pyproject.toml for inference parity.
!pip install -q \
    "torch>=2.12.0" \
    "transformers>=5.8.1" \
    "datasets>=4.8.5" \
    "evaluate>=0.4.6" \
    "accelerate>=1.13.0" \
    "scikit-learn>=1.8.0" \
    "pydantic>=2.13.4" \
    "httpx>=0.28.1"
    "python-dotenv>=1.0.0"

import torch
print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 67.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 62.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━

In [2]:
# Mount Google Drive — data in, artifacts out.
# If you'd rather work fully ephemerally, comment this out and use /content paths.
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive')

from pathlib import Path
WORK_DIR = Path('/content/drive/MyDrive/maintainers-copilot')
DATA_DIR = WORK_DIR / 'data'
ARTIFACTS_DIR = WORK_DIR / 'artifacts' / 'classifier'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_DIR =', WORK_DIR)

Mounted at /content/drive
WORK_DIR = /content/drive/MyDrive/maintainers-copilot


In [10]:
# Optional: GitHub PAT raises rate limit from 60 → 5000 req/hour.
# Stored as a Colab secret OR in .env file at project root.
import os
from pathlib import Path

def load_secrets():
    """Load from Colab web secrets or .env file."""
    # Attempt 1: Colab web secrets
    try:
        from google.colab import userdata
        github = userdata.get("GITHUB_TOKEN")
        gemini = userdata.get("GEMINI_API_KEY")
        if github:
            os.environ["GITHUB_TOKEN"] = github
        if gemini:
            os.environ["GEMINI_API_KEY"] = gemini
        if github or gemini:
            return "Colab web secrets"
    except Exception:
        pass
    
    # Attempt 2: Read .env file directly (no dotenv needed)
    env_file = Path("/home/user/workplace/aie_sef_bootcamp/project7/.env")
    if env_file.exists():
        try:
            with env_file.open() as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#"):
                        if "=" in line:
                            key, value = line.split("=", 1)
                            key, value = key.strip(), value.strip()
                            if key in ("GITHUB_TOKEN", "GEMINI_API_KEY"):
                                os.environ[key] = value
            return f".env file at {env_file}"
        except Exception as e:
            return f"error reading .env: {e}"
    
    return "not found (using anonymous limits)"

source = load_secrets()
print(f"✓ Secrets loaded from: {source}")
print(f"  GITHUB_TOKEN: {"set ✓" if os.environ.get("GITHUB_TOKEN") else "not set"}")
print(f"  GEMINI_API_KEY: {"set ✓" if os.environ.get("GEMINI_API_KEY") else "not set"}")


⚠ .env not found at /.env


## 1. Fetch MONAI closed issues from GitHub

In [ ]:
import json
import time
from datetime import datetime
from typing import Any

import httpx

REPO = 'Project-MONAI/MONAI'
TARGET_LABELS = ('bug', 'feature request', 'documentation', 'questions')
RAW_PATH = DATA_DIR / 'raw_issues.jsonl'


def fetch_closed_issues(repo: str, force: bool = False) -> list[dict[str, Any]]:
    """Page through GitHub API; cache to RAW_PATH so we don't refetch on rerun."""
    if RAW_PATH.exists() and not force:
        with RAW_PATH.open() as fh:
            rows = [json.loads(line) for line in fh]
        print(f'cache hit: {len(rows)} issues from {RAW_PATH}')
        return rows

    headers = {'Accept': 'application/vnd.github+json'}
    if token := os.environ.get('GITHUB_TOKEN'):
        headers['Authorization'] = f'Bearer {token}'

    issues: list[dict[str, Any]] = []
    page = 1
    with httpx.Client(timeout=30.0, headers=headers) as client:
        while True:
            r = client.get(
                f'https://api.github.com/repos/{repo}/issues',
                params={'state': 'closed', 'per_page': 100, 'page': page},
            )
            r.raise_for_status()
            batch = r.json()
            if not batch:
                break
            # Skip PRs (GitHub API returns them in /issues with a pull_request field).
            issues.extend(i for i in batch if 'pull_request' not in i)
            print(f'page {page}: +{len(batch)}  total={len(issues)}')
            page += 1
            time.sleep(0.2)  # be polite

    # Slim & save.
    slim = [
        {
            'id': i['id'],
            'number': i['number'],
            'title': i['title'],
            'body': i.get('body'),
            'labels': [l['name'] for l in i.get('labels', [])],
            'created_at': i['created_at'],
            'closed_at': i['closed_at'],
        }
        for i in issues
    ]
    with RAW_PATH.open('w') as fh:
        for row in slim:
            fh.write(json.dumps(row) + '\n')
    print(f'saved {len(slim)} issues → {RAW_PATH}')
    return slim


raw_issues = fetch_closed_issues(REPO)

## 2. Label mapping — 4 GitHub labels → 3 canonical classes

**Merge rationale (also in `docs/DECISIONS.md`):** `documentation` has only 28 closed issues → ~5 test examples after a 70/15/15 split. Per-class F1 on 5 examples is statistical noise (1 misclassification ≈ 20-pt swing). For a maintainer, the routing decision for `documentation` and `questions` is identical ("point user to the right doc/FAQ"), so merging them into `support` is semantically valid and gives a balanced, defensible 3-class problem.

In [13]:
from collections import Counter

# Get all labels from every issue, not just those in our TARGET_LABELS set
all_raw_labels = [lbl for issue in raw_issues for lbl in issue.get('labels', [])]
label_distribution = Counter(all_raw_labels)

print(f"Total unique labels found: {len(label_distribution)}")
print("\nTop 50 most frequent labels (look for variations of bug, feature, doc, question):")
print("-" * 60)
for lbl, count in label_distribution.most_common(50):
    print(f"{lbl:<40} | {count}")

NameError: name 'raw_issues' is not defined

In [ ]:
from typing import Any

from collections import Counter

# Priority mapping: support > bug > feature
PRIORITY = {'support': 0, 'bug': 1, 'feature': 2}

LABEL_MAP: dict[str, str] = {
    'bug': 'bug',
    'Feature request': 'feature',
    'feature request': 'feature',
    'enhancement': 'feature',
    'documentation': 'support',
    'question': 'support',
    'questions': 'support',
}
CLASS_NAMES: tuple[str, ...] = ('bug', 'feature', 'support')
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
_TARGET_LABELS = frozenset(LABEL_MAP.keys())

def resolve_label(labels: list[str]) -> str | None:
    """Return canonical class based on priority if multiple are present."""
    mapped = {LABEL_MAP[lbl] for lbl in labels if lbl in _TARGET_LABELS}
    if not mapped:
        return None
    
    # If multiple labels exist, pick the one with the highest priority (lowest rank index)
    # Priority: support (0) > bug (1) > feature (2)
    return min(mapped, key=lambda x: PRIORITY[x])

def build_row(raw: dict[str, Any]) -> dict[str, Any] | None:
    label = resolve_label(raw['labels'])
    if label is None or raw.get('closed_at') is None:
        return None
    text = f"{raw['title']}\n\n{raw.get('body') or ''}".strip()
    return {
        'id': raw['id'],
        'number': raw['number'],
        'text': text,
        'label': label,
        'label_idx': CLASS_TO_IDX[label],
        'closed_at': raw['closed_at'],
    }

# Build labeled rows with priority logic.
labeled = [r for r in (build_row(i) for i in raw_issues) if r is not None]
class_counts = Counter(r['label'] for r in labeled)

print(f'After resolve_label (Priority: support > bug > feature) ({len(labeled)} usable issues):')
for c in CLASS_NAMES:
    print(f'  {c:<10} {class_counts[c]}')

NameError: name 'Any' is not defined

## 3. Stratified time-aware split (70/15/15)

Most recent 15% by `closed_at` → **test** (prevents temporal leakage; the model must generalise to future-shaped issues).
Remaining 85% → stratified by class → **train (82.4%) + val (17.6%)** of the remainder = ~15% of total.

In [ ]:
import random
from collections import defaultdict

RANDOM_SEED = 42
TEST_FRACTION = 0.15
VAL_OF_REMAINING = 0.15 / 0.85

rng = random.Random(RANDOM_SEED)
labeled_sorted = sorted(labeled, key=lambda r: r['closed_at'])

# Time-aware test cut.
test_cut = int(len(labeled_sorted) * (1 - TEST_FRACTION))
remaining, test = labeled_sorted[:test_cut], labeled_sorted[test_cut:]

# Verify temporal invariant.
assert max(r['closed_at'] for r in remaining) <= min(r['closed_at'] for r in test), \
    'temporal leakage: train/val overlaps test in time'

# Stratified val sample from remaining.
by_class: dict[str, list[dict[str, Any]]] = defaultdict(list)
for r in remaining:
    by_class[r['label']].append(r)

train: list[dict[str, Any]] = []
val: list[dict[str, Any]] = []
for cls, rows in by_class.items():
    rng.shuffle(rows)
    n_val = max(1, int(len(rows) * VAL_OF_REMAINING))
    val.extend(rows[:n_val])
    train.extend(rows[n_val:])

rng.shuffle(train)
rng.shuffle(val)

for name, split in (('train', train), ('val', val), ('test', test)):
    cc = Counter(r['label'] for r in split)
    print(f'{name:<5} n={len(split):<5}  ' + '  '.join(f'{c}={cc[c]}' for c in CLASS_NAMES))

# Persist for reproducibility.
for name, split in (('train', train), ('val', val), ('test', test)):
    path = DATA_DIR / f'{name}.jsonl'
    with path.open('w') as fh:
        for row in split:
            fh.write(json.dumps(row) + '\n')
    print(f'saved → {path}')

## 4. Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf(split: list[dict[str, Any]]) -> Dataset:
    ds = Dataset.from_dict({
        'text': [r['text'] for r in split],
        'label': [r['label_idx'] for r in split],
    })
    return ds.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH, padding=False),
        batched=True,
    )


train_ds = to_hf(train)
val_ds = to_hf(val)
print(train_ds, val_ds, sep='\n')

## 5. Train DistilBERT (3-class head, EarlyStopping)

In [ ]:
import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

OUTPUT_DIR = Path('/content/artifacts/classifier')
BEST_DIR = OUTPUT_DIR / 'best'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    id2label={i: c for i, c in enumerate(CLASS_NAMES)},
    label2id=CLASS_TO_IDX,
)

f1_metric = evaluate.load('f1')


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=preds, references=labels, average='macro')


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 6. Save best checkpoint + write `model_card.json`

**The model card is the contract with the backend.** `load_classifier()` reads `classes`, `model_sha256`, `version`, `hyperparameters.max_length` and refuses to boot on any drift.

In [ ]:
import hashlib
from datetime import UTC

BEST_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))


def sha256_weights(directory: Path) -> str:
    """MUST match the algorithm in backend/app/ml/classifier.py::_sha256_dir_weights."""
    h = hashlib.sha256()
    matched: list[Path] = []
    for pat in ('*.safetensors', '*.bin'):
        matched.extend(sorted(directory.glob(pat)))
    for p in matched:
        with p.open('rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                h.update(chunk)
    return f'sha256:{h.hexdigest()}'


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return f'sha256:{h.hexdigest()}'


eval_results = trainer.evaluate()
val_f1 = float(eval_results.get('eval_f1', 0.0))

model_sha256 = sha256_weights(BEST_DIR)
train_path = DATA_DIR / 'train.jsonl'
train_sha256 = sha256_file(train_path)

model_card = {
    'architecture': MODEL_NAME,
    'num_labels': len(CLASS_NAMES),
    'classes': list(CLASS_NAMES),
    'class_to_idx': CLASS_TO_IDX,
    'hyperparameters': {
        'learning_rate': training_args.learning_rate,
        'per_device_train_batch_size': training_args.per_device_train_batch_size,
        'num_train_epochs': training_args.num_train_epochs,
        'weight_decay': training_args.weight_decay,
        'warmup_ratio': training_args.warmup_ratio,
        'max_length': MAX_LENGTH,
    },
    'freeze_policy': 'all layers unfrozen — full fine-tune of DistilBERT',
    'training_data_sha256': train_sha256,
    'training_data_size': {'train': len(train), 'val': len(val), 'test': len(test)},
    'metrics': {'val_f1_macro': val_f1, 'raw_eval': eval_results},
    'model_sha256': model_sha256,
    'trained_at': datetime.now(UTC).isoformat(),
    'version': '1.0.0',
    'dataset_repo': REPO,
}

(BEST_DIR / 'model_card.json').write_text(json.dumps(model_card, indent=2))
print(json.dumps(model_card, indent=2))

## 6a. Evaluate DistilBERT on test split

Produces the DL row in the three-way comparison. Uses the same `test` split as the other two baselines.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
import time

test_ds = to_hf(test)
test_labels_idx = [r['label_idx'] for r in test]

t0 = time.perf_counter()
dl_preds_output = trainer.predict(test_ds)
dl_latency_ms = (time.perf_counter() - t0) / len(test) * 1000  # avg per sample

dl_preds = np.argmax(dl_preds_output.predictions, axis=-1).tolist()
dl_acc = accuracy_score(test_labels_idx, dl_preds)
dl_macro_f1 = f1_score(test_labels_idx, dl_preds, average='macro')
dl_per_class_f1 = f1_score(test_labels_idx, dl_preds, average=None).tolist()

print(f'DistilBERT  accuracy  : {dl_acc:.4f}')
print(f'DistilBERT  macro-F1  : {dl_macro_f1:.4f}')
print(f'Per-class F1: {dict(zip(CLASS_NAMES, dl_per_class_f1))}')
print(f'Latency (avg/sample)  : {dl_latency_ms:.2f} ms')
print()
print(classification_report(test_labels_idx, dl_preds, target_names=CLASS_NAMES))
print('Confusion matrix (row=true, col=pred):')
print(confusion_matrix(test_labels_idx, dl_preds))


## 6b. Classical ML baseline — TF-IDF + Logistic Regression

Trains on the same split. No GPU needed. Serves as the lower-bound comparison.

Defended in `docs/DECISIONS.md`: TF-IDF bigrams capture surface patterns well for short issue titles; LogReg is fast and interpretable; together they represent the pre-DL state of the art for text classification.

In [ ]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

train_texts = [r['text'] for r in train]
train_labels_list = [r['label_idx'] for r in train]
test_texts = [r['text'] for r in test]

ml_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50_000,
        sublinear_tf=True,
        strip_accents='unicode',
    )),
    ('lr', LogisticRegression(
        C=1.0,
        max_iter=1_000,
        random_state=RANDOM_SEED,
        class_weight='balanced',
    )),
])

print('Training TF-IDF + LogReg...')
ml_pipeline.fit(train_texts, train_labels_list)

t0 = time.perf_counter()
ml_preds = ml_pipeline.predict(test_texts).tolist()
ml_latency_ms = (time.perf_counter() - t0) / len(test) * 1000

ml_acc = accuracy_score(test_labels_idx, ml_preds)
ml_macro_f1 = f1_score(test_labels_idx, ml_preds, average='macro')
ml_per_class_f1 = f1_score(test_labels_idx, ml_preds, average=None).tolist()

print(f'TF-IDF+LR  accuracy  : {ml_acc:.4f}')
print(f'TF-IDF+LR  macro-F1  : {ml_macro_f1:.4f}')
print(f'Per-class F1: {dict(zip(CLASS_NAMES, ml_per_class_f1))}')
print(f'Latency (avg/sample) : {ml_latency_ms:.4f} ms')
print()
print(classification_report(test_labels_idx, ml_preds, target_names=CLASS_NAMES))

# Persist for reproducibility audit (not used in production).
ml_pkl_path = OUTPUT_DIR.parent / 'classical_ml' / 'pipeline.pkl'
ml_pkl_path.parent.mkdir(parents=True, exist_ok=True)
with ml_pkl_path.open('wb') as fh:
    pickle.dump(ml_pipeline, fh)
print(f'Saved -> {ml_pkl_path}')


## 6c. LLM baseline — Gemini few-shot

Uses `gemini-2.0-flash` with a `k=5`-shot prompt per class (15 examples total in context).
Evaluated on a configurable sample of the test split (`GEMINI_EVAL_SAMPLE`; default 100) to control cost. Records accuracy, macro-F1, per-class F1, average latency, and estimated cost per 1 K predictions.

**Requires `GEMINI_API_KEY` in Colab secrets** (Colab → Secrets → `GEMINI_API_KEY`). If missing, the LLM baseline is skipped and the comparison uses only two models.

In [ ]:
import random as _random

# Load Gemini API key from Colab secrets.
GEMINI_API_KEY = ''
try:
    from google.colab import userdata  # type: ignore[import-not-found]
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') or ''
    print('GEMINI_API_KEY loaded from Colab secrets')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not GEMINI_API_KEY:
    print('WARNING: GEMINI_API_KEY not set -- LLM baseline will be skipped')

GEMINI_MODEL = 'gemini-2.0-flash'
GEMINI_EVAL_SAMPLE = 100   # lower for cost control; set to len(test) for full eval
# Flash pricing 2025: $0.10/1M input, $0.40/1M output
GEMINI_COST_PER_IN_TOK  = 0.10 / 1_000_000
GEMINI_COST_PER_OUT_TOK = 0.40 / 1_000_000

# Build k-shot prompt (k examples per class from train set).
K_SHOT = 5
few_shot_examples: list[dict] = []
for cls in CLASS_NAMES:
    cls_rows = [r for r in train if r['label'] == cls]
    _rng = _random.Random(RANDOM_SEED)
    _rng.shuffle(cls_rows)
    for row in cls_rows[:K_SHOT]:
        few_shot_examples.append({'text': row['text'][:400].strip(), 'label': cls})
_random.Random(RANDOM_SEED + 1).shuffle(few_shot_examples)

CLASSIFY_SYSTEM = (
    'You are an issue classifier for open-source repositories.\n'
    'Classify the GitHub issue into exactly one of: bug, feature, support.\n\n'
    '  bug     -- A defect, crash, regression, or unexpected behaviour.\n'
    '  feature -- A request for new functionality or an enhancement.\n'
    '  support -- A question, docs gap, or request for help.\n\n'
    'Reply with ONLY the single class name (lowercase, no punctuation, nothing else).'
)


def _few_shot_contents(text: str) -> list[dict]:
    contents = []
    for ex in few_shot_examples:
        contents.append({'role': 'user',  'parts': [{'text': ex['text']}]})
        contents.append({'role': 'model', 'parts': [{'text': ex['label']}]})
    contents.append({'role': 'user', 'parts': [{'text': text[:600]}]})
    return contents


def classify_with_gemini(text: str, client: httpx.Client) -> tuple[str, float, int, int]:
    """Returns (label, latency_ms, input_tokens, output_tokens)."""
    body = {
        'system_instruction': {'parts': [{'text': CLASSIFY_SYSTEM}]},
        'contents': _few_shot_contents(text),
        'generationConfig': {'temperature': 0.0, 'maxOutputTokens': 10},
    }
    t0 = time.perf_counter()
    r = client.post(
        f'https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent',
        params={'key': GEMINI_API_KEY},
        json=body,
        timeout=30.0,
    )
    latency_ms = (time.perf_counter() - t0) * 1000
    r.raise_for_status()
    data = r.json()
    raw = data['candidates'][0]['content']['parts'][0]['text'].strip().lower()
    label = raw if raw in CLASS_NAMES else 'support'   # fallback on hallucination
    usage = data.get('usageMetadata', {})
    return label, latency_ms, usage.get('promptTokenCount', 0), usage.get('candidatesTokenCount', 0)


# Run evaluation.
llm_results: list[dict] = []
llm_acc = llm_macro_f1 = llm_avg_latency = llm_cost_per_1k = 0.0
llm_per_class_f1: list[float] = [0.0] * len(CLASS_NAMES)

if GEMINI_API_KEY:
    test_sample = test[:GEMINI_EVAL_SAMPLE]
    print(f'Evaluating {GEMINI_MODEL} on {len(test_sample)} examples ({K_SHOT}-shot per class)...')

    with httpx.Client() as client:
        for i, row in enumerate(test_sample):
            pred, lat, in_tok, out_tok = classify_with_gemini(row['text'], client)
            llm_results.append({
                'true': row['label'], 'pred': pred,
                'latency_ms': lat, 'input_tokens': in_tok, 'output_tokens': out_tok,
            })
            if (i + 1) % 10 == 0:
                print(f'  {i + 1}/{len(test_sample)}')
            time.sleep(0.1)

    llm_true_idx = [CLASS_TO_IDX[r['true']] for r in llm_results]
    llm_pred_idx = [CLASS_TO_IDX.get(r['pred'], 2) for r in llm_results]
    llm_acc          = accuracy_score(llm_true_idx, llm_pred_idx)
    llm_macro_f1     = f1_score(llm_true_idx, llm_pred_idx, average='macro')
    llm_per_class_f1 = f1_score(llm_true_idx, llm_pred_idx, average=None).tolist()
    llm_avg_latency  = sum(r['latency_ms']   for r in llm_results) / len(llm_results)
    avg_in  = sum(r['input_tokens']  for r in llm_results) / len(llm_results)
    avg_out = sum(r['output_tokens'] for r in llm_results) / len(llm_results)
    llm_cost_per_1k  = (avg_in * GEMINI_COST_PER_IN_TOK + avg_out * GEMINI_COST_PER_OUT_TOK) * 1_000

    print(f'\n{GEMINI_MODEL}  accuracy  : {llm_acc:.4f}')
    print(f'{GEMINI_MODEL}  macro-F1  : {llm_macro_f1:.4f}')
    print(f'Per-class F1: {dict(zip(CLASS_NAMES, llm_per_class_f1))}')
    print(f'Avg latency            : {llm_avg_latency:.1f} ms')
    print(f'Cost per 1K preds      : ${llm_cost_per_1k:.4f}')
    print()
    print(classification_report(llm_true_idx, llm_pred_idx, target_names=CLASS_NAMES))
else:
    print('LLM baseline skipped -- GEMINI_API_KEY not set.')


## 6d. Three-way comparison → `eval_report.json`

Selects the deployment winner by test macro-F1 (primary) and latency (tiebreaker). Writes `eval_report.json` to Drive — the backend's `eval/run_classification_eval.py` loads this from MinIO for CI gating.

**Decision rubric (also in `docs/DECISIONS.md`):**
- DistilBERT: highest expected F1; ~150 ms/sample on CPU; zero per-call cost.
- TF-IDF+LR: fastest (<1 ms), lower semantic ceiling.
- Gemini few-shot: high F1 zero-shot but ~800 ms/call and $/1K cost — too slow and costly for an interactive endpoint.

In [ ]:
from datetime import UTC

THRESHOLDS = {'macro_f1': 0.78, 'per_class_f1_min': 0.70}

candidates: dict[str, dict] = {
    'distilbert': {
        'accuracy': float(dl_acc),
        'macro_f1': float(dl_macro_f1),
        'per_class_f1': {c: float(f) for c, f in zip(CLASS_NAMES, dl_per_class_f1)},
        'avg_latency_ms': float(dl_latency_ms),
        'cost_per_1k_predictions': 0.0,
        'eval_sample_size': len(test),
    },
    'tfidf_logreg': {
        'accuracy': float(ml_acc),
        'macro_f1': float(ml_macro_f1),
        'per_class_f1': {c: float(f) for c, f in zip(CLASS_NAMES, ml_per_class_f1)},
        'avg_latency_ms': float(ml_latency_ms),
        'cost_per_1k_predictions': 0.0,
        'eval_sample_size': len(test),
    },
}
if GEMINI_API_KEY and llm_results:
    candidates[f'gemini_{GEMINI_MODEL.replace("-", "_")}'] = {
        'accuracy': float(llm_acc),
        'macro_f1': float(llm_macro_f1),
        'per_class_f1': {c: float(f) for c, f in zip(CLASS_NAMES, llm_per_class_f1)},
        'avg_latency_ms': float(llm_avg_latency),
        'cost_per_1k_predictions': float(llm_cost_per_1k),
        'eval_sample_size': len(llm_results),
    }

winner_name = max(candidates, key=lambda k: candidates[k]['macro_f1'])
winner_metrics = candidates[winner_name]

# Print comparison table.
print('=== THREE-WAY COMPARISON ===\n')
print(f"{'Model':<30} {'Accuracy':>10} {'Macro-F1':>10} {'Latency ms':>12} {'$/1K':>10}")
print('-' * 75)
for name, m in sorted(candidates.items(), key=lambda kv: kv[1]['macro_f1'], reverse=True):
    marker = ' <- WINNER' if name == winner_name else ''
    print(
        f"{name:<30} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f} "
        f"{m['avg_latency_ms']:>12.2f} ${m['cost_per_1k_predictions']:>9.4f}{marker}"
    )

threshold_pass = (
    winner_metrics['macro_f1'] >= THRESHOLDS['macro_f1']
    and all(v >= THRESHOLDS['per_class_f1_min'] for v in winner_metrics['per_class_f1'].values())
)

eval_report = {
    'generated_at': datetime.now(UTC).isoformat(),
    'dataset_repo': REPO,
    'test_split_size': len(test),
    'llm_eval_sample_size': len(llm_results) if llm_results else 0,
    'classes': list(CLASS_NAMES),
    'models': candidates,
    'winner': {
        'model': winner_name,
        'rationale': (
            f"Highest macro-F1 ({winner_metrics['macro_f1']:.4f}) on the time-aware test split. "
            'DistilBERT is self-hosted (zero per-call cost) and its latency is acceptable '
            'for an interactive endpoint behind a model-server container.'
        ),
        'metrics': winner_metrics,
    },
    'deployment_decision': winner_name,
    'thresholds': THRESHOLDS,
    'threshold_pass': threshold_pass,
}

EVAL_REPORT_PATH = WORK_DIR / 'artifacts' / 'eval_report.json'
EVAL_REPORT_PATH.write_text(json.dumps(eval_report, indent=2))
print(f'\neval_report.json -> {EVAL_REPORT_PATH}')
print(f'threshold_pass   : {threshold_pass}')
if not threshold_pass:
    print('WARNING: winning model is below committed thresholds -- inspect per-class F1 before deploying')


## 7. Persist artifacts to Drive (zip + copy)

In [ ]:
import shutil

# Copy the unzipped best/ to Drive -- what load_classifier() will read.
drive_best = ARTIFACTS_DIR / 'best'
if drive_best.exists():
    shutil.rmtree(drive_best)
shutil.copytree(BEST_DIR, drive_best)

# Also zip for portability (download / scp / upload to MinIO).
zip_base = ARTIFACTS_DIR / 'classifier_best'
shutil.make_archive(str(zip_base), 'zip', root_dir=BEST_DIR.parent, base_dir=BEST_DIR.name)

# Copy eval_report.json alongside the model artifacts.
local_eval_report = WORK_DIR / 'artifacts' / 'eval_report.json'
if local_eval_report.exists():
    shutil.copy2(local_eval_report, ARTIFACTS_DIR / 'eval_report.json')
    print(f'eval_report.json -> {ARTIFACTS_DIR / "eval_report.json"}')
else:
    print('WARNING: eval_report.json not found -- run Section 6d first')

print('Persisted to Drive:')
for p in sorted(drive_best.iterdir()):
    print(f'  {p.name:<30} {p.stat().st_size:>12} bytes')
print(f'Zip archive: {zip_base}.zip ({Path(str(zip_base) + ".zip").stat().st_size} bytes)')


## 8. Round-trip verify — simulate `load_classifier()`

This re-runs **the exact checks** the backend will perform at boot. If this cell fails, the backend will refuse to start with the same error. Run it before you commit the artifact.

In [ ]:
EXPECTED_CLASSES = ('bug', 'feature', 'support')  # backend contract

card = json.loads((drive_best / 'model_card.json').read_text())

assert 'classes' in card, 'card missing classes field'
assert tuple(card['classes']) == EXPECTED_CLASSES, (
    f'class-set drift: card={card["classes"]} vs backend={list(EXPECTED_CLASSES)}'
)

assert card.get('model_sha256', '').startswith('sha256:'), 'card missing model_sha256'

actual = sha256_weights(drive_best)
assert actual == card['model_sha256'], f'sha256 mismatch\n  expected: {card["model_sha256"]}\n  actual:   {actual}'

assert (drive_best / 'tokenizer.json').exists() or (drive_best / 'tokenizer_config.json').exists(), (
    'tokenizer files missing — AutoTokenizer.from_pretrained() will fail'
)

print('OK — artifact passes every check load_classifier() will run.')
print(f'   classes      : {card["classes"]}')
print(f'   val_f1_macro : {card["metrics"]["val_f1_macro"]:.4f}')
print(f'   model_sha256 : {card["model_sha256"][:24]}…')

## Deploy to backend

1. **Download** `artifacts/classifier/best/` from Drive (or fetch directly inside docker via gdown / boto3 / MinIO).
2. **Place** at `backend/artifacts/classifier/best/` (gitignored — never commit weights).
3. **Start the stack:** `docker-compose up`. The model-server lifespan will call `load_classifier(Path('artifacts/classifier/best'))` and verify the same checks Section 8 just passed.
4. **Smoke test:** `curl -X POST localhost:8000/classify -d '{"text":"crash on empty tensor"}'` → `{"label":"bug", ...}`.